Canvas announcement

You will notice the files for the ENSU levels were uploaded last night, so you can skip bullet one. No imputation is needed since I have managed the population-weights on my side.

Note, however, that when you attempt to create your composite index (z-score variables, sum, then divide by 3; easy since all variable directions are consistent), you will run into a sampling issue with the survey data. Not all municipalities were surveyed in 2025. Replace missing values with zero and create a second variable that serves as a non_sampled indicator. This approach allows you to rank all municipalities and correct the index bias induced by the zero insertion. Do the same for any PCA/LDA approaches. 

Once you produce ranks, see how each method stacks up. At least one of the three above should get you a very strong first stage for 43. In my best approach so far (and yours may be better!), that leaves 18 strategic selections that cannot be explained by the ranks (they are very far away from the lower tail of the distribution). 

We will be circling, checking on progress from yesterday. I would recommend front loading today's work before noon with a focus on the ranking calculations, and then jump back to covariates. We will need the output of both work streams tomorrow. 

# Briefing memo instructions

Tuesday, September 1: completing the criteria data and building the candidates

Assemble perception (criterion 3): ENSU levels and changes where observed, ENVIPE state fall- back elsewhere, coverage and imputation choices written down as made; merge into the criteria cross-section

Fill the DIRECTION map for every criteria variable; run the printed sanity extremes; whole-team
sign-off before anything downstream

Build the composite candidate, equal-weight and inverse-variance versions

Build the PCA candidate; verify the sign alignment against the composite

Build the LDA candidate trained on the 53; verify in code that the 8 strategic municipalities are excluded from training

Produce ranks; confirm the rank-1-equals-worst assertions pass; eyeball the top-53 and name thecases you expected to see

## Composite index (Z score)

In [34]:
import pandas as pd
import numpy as np
from scipy import stats

treated = pd.read_csv("data/treated61-2.csv")

# Load data
xlsx = pd.read_excel("data/cs_municipal_2025.xlsx")
csv = pd.read_csv("data/cs_municipal_2025_popweighted.csv")


print(csv["CVEGEO"].nunique())
print(csv["municipio"].nunique())


print(xlsx.head())
print(xlsx["cve_mun_completa"].nunique())


2486
2336
   clave_ent         entidad  cve_mun_completa       municipio  \
0          1  Aguascalientes              1001  Aguascalientes   
1          1  Aguascalientes              1002        Asientos   
2          1  Aguascalientes              1003        Calvillo   
3          1  Aguascalientes              1004          Cos√≠o   
4          1  Aguascalientes              1005   Jes√∫s Mar√≠a   

   homicidio_doloso_prom_diario  robo_veh_violencia_prom_diario  \
0                      0.112329                        0.128767   
1                      0.008219                        0.027397   
2                      0.000000                        0.005479   
3                      0.005479                        0.019178   
4                      0.041096                        0.043836   

   nom_mun_envipe  n_muestra_percepcion  pct_inseguro_municipio  
0  AGUASCALIENTES                1305.0                0.402567  
1        ASIENTOS                 106.0                0.6

In [27]:

# Rename merge keys to match
xlsx = xlsx.rename(columns={"cve_mun_completa": "CVEGEO"})

# Merge — keep all municipalities from xlsx (left join)
df = xlsx.merge(
    csv[["CVEGEO", "n_inseguro_popw"]],
    on="CVEGEO",
    how="left"
)

# Flag municipalities not surveyed (missing perception data)
df["non_sampled"] = df["n_inseguro_popw"].isna().astype(int)

# Replace missing perception with 0
df["n_inseguro_popw"] = df["n_inseguro_popw"].fillna(0)

# Z-score the three criteria
# All three are directionally consistent: higher = worse
criteria = {
    "z_homicidio": "homicidio_doloso_prom_diario",
    "z_robo_veh":  "robo_veh_violencia_prom_diario",
    "z_inseguro":  "n_inseguro_popw",
}

for z_col, raw_col in criteria.items():
    df[z_col] = stats.zscore(df[raw_col], nan_policy="omit")

# Composite: sum z-scores, divide by 3
df["composite"] = (df["z_homicidio"] + df["z_robo_veh"] + df["z_inseguro"]) / 3

# Rank: 1 = worst (highest composite)
df["rank_composite"] = df["composite"].rank(ascending=False, method="min").astype(int)

# Sanity check: print extremes
print("=== TOP 5 (worst) ===")
print(df.nsmallest(5, "rank_composite")[
    ["CVEGEO", "municipio", "entidad", "composite", "rank_composite", "non_sampled"]
])

print("\n=== BOTTOM 5 (best) ===")
print(df.nlargest(5, "rank_composite")[
    ["CVEGEO", "municipio", "entidad", "composite", "rank_composite", "non_sampled"]
])

print(f"\nTotal municipalities: {len(df)}")
print(f"Non-sampled (perception=0): {df['non_sampled'].sum()}")

# Export
df.to_csv("composite_index.csv", index=False)
print("\nSaved to composite_index.csv")



=== TOP 5 (worst) ===
      CVEGEO            municipio          entidad  composite  rank_composite  \
1899   25006            Culiac√°n          Sinaloa  21.471978               1   
14      2004              Tijuana  Baja California  14.887538               2   
704    15033  Ecatepec de Morelos          M√©xico  10.891724               3   
244     8037              Ju√°rez        Chihuahua  10.366703               4   
1701   21114               Puebla           Puebla   8.422815               5   

      non_sampled  
1899            0  
14              0  
704             0  
244             0  
1701            0  

=== BOTTOM 5 (best) ===
    CVEGEO         municipio               entidad  composite  rank_composite  \
40    5005           Candela  Coahuila de Zaragoza  -0.209326            1826   
42    5007  Cuatro Ci√©negas  Coahuila de Zaragoza  -0.209326            1826   
43    5008          Escobedo  Coahuila de Zaragoza  -0.209326            1826   
46    5011    General 

In [28]:
print(treated.head())
print('-----DF----')
print(df.head())

   CVEGEO          entidad           municipio  gov_inclusion_code
0    2004  Baja California             Tijuana  homdol_rvcv_percep
1    6002           Colima              Colima              homdol
2   12001         Guerrero  Acapulco de Juárez         homdol_rvcv
3   25006          Sinaloa            Culiacán  homdol_rvcv_percep
4   26018           Sonora              Cajeme              homdol
-----DF----
   clave_ent         entidad  CVEGEO       municipio  \
0          1  Aguascalientes    1001  Aguascalientes   
1          1  Aguascalientes    1002        Asientos   
2          1  Aguascalientes    1003        Calvillo   
3          1  Aguascalientes    1004          Cos√≠o   
4          1  Aguascalientes    1005   Jes√∫s Mar√≠a   

   homicidio_doloso_prom_diario  robo_veh_violencia_prom_diario  \
0                      0.112329                        0.128767   
1                      0.008219                        0.027397   
2                      0.000000                 

In [29]:
treated["CVEGEO"] = treated["CVEGEO"].astype(str).str.zfill(5)
df["CVEGEO"] = df["CVEGEO"].astype(str).str.zfill(5)  # <- add this

df["treated"] = df["CVEGEO"].isin(treated["CVEGEO"]).astype(int)


print("Sample df CVEGEOs:", df["CVEGEO"].head(3).tolist())
print("Sample treated CVEGEOs:", treated["CVEGEO"].head(3).tolist())


top61 = df.nsmallest(100, "rank_composite")[
    ["CVEGEO", "municipio", "entidad", "composite", "rank_composite", "non_sampled", "treated"]
]
top61.to_csv("top61-zscore.csv", index=False)


print("RANK 1-61")
print(top61.to_string())
print(f"\nTreated in top 61: {top61['treated'].sum()} / 61")

Sample df CVEGEOs: ['01001', '01002', '01003']
Sample treated CVEGEOs: ['02004', '06002', '12001']
RANK 1-61
     CVEGEO                     municipio                          entidad  composite  rank_composite  non_sampled  treated
1899  25006                     Culiac√°n                          Sinaloa  21.471978               1            0        1
14    02004                       Tijuana                  Baja California  14.887538               2            0        1
704   15033           Ecatepec de Morelos                          M√©xico  10.891724               3            0        1
244   08037                       Ju√°rez                        Chihuahua  10.366703               4            0        1
1701  21114                        Puebla                           Puebla   8.422815               5            0        1
350   11020                         Le√≥n                       Guanajuato   8.038209               6            0        1
377   12001           A

## PCA

In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- Prep features (same as composite) ---
features = ["homicidio_doloso_prom_diario", "robo_veh_violencia_prom_diario", "n_inseguro_popw"]


## DF TO USE 
df_pca = df[["CVEGEO", "municipio", "entidad", "non_sampled", "treated"] + features].copy()

# Z-score
scaler = StandardScaler()
df_pca[features] = scaler.fit_transform(df_pca[features])

# --- PCA ---
pca = PCA(n_components=3) # fit PCA on 3 z-scored vars
pca.fit(df_pca[features])


,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",3
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD 

In [18]:
# print hhow much of the total variation of each component explains 
# PC1 = __ means that one dimension captures ___ of what is happening across three criteria
print("=== Variance explained by each component ===")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.3f} ({var*100:.1f}%)")

print("\n=== PC1 loadings (sign check) ===")
for feat, loading in zip(features, pca.components_[0]):
    print(f"  {feat}: {loading:.4f}")


=== Variance explained by each component ===
  PC1: 0.763 (76.3%)
  PC2: 0.136 (13.6%)
  PC3: 0.101 (10.1%)

=== PC1 loadings (sign check) ===
  homicidio_doloso_prom_diario: 0.5871
  robo_veh_violencia_prom_diario: 0.5609
  n_inseguro_popw: 0.5837


In [19]:
# PC1 scores
# computes each municipality's PC1 score, which is its position along the first principal component
df_pca["pc1_score"] = pca.fit_transform(df_pca[features])[:, 0]

# Sign alignment: PC1 should correlate positively with composite
# (higher score = worse). Flip if needed.
corr = df_pca["pc1_score"].corr(df["composite"])
print(f"\nCorrelation of PC1 with composite: {corr:.4f}")
if corr < 0:
    print("Flipping PC1 sign to align with composite direction.")
    df_pca["pc1_score"] = -df_pca["pc1_score"]

# Rank: 1 = worst
# rank by PC1 score
df_pca["rank_pca"] = df_pca["pc1_score"].rank(ascending=False, method="min").astype(int)

# --- Coverage check ---
# pull 61 worst ranked municipalities -- this is the same as the composite
top61_pca = df_pca.nsmallest(61, "rank_pca")[
    ["CVEGEO", "municipio", "entidad", "pc1_score", "rank_pca", "non_sampled", "treated"]
]
print("\nRANK 1-61 (PCA)")
print(top61_pca)
print(f"\nTreated in top 61 (PCA): {top61_pca['treated'].sum()} / 61")

# --- Compare composite vs PCA ranks for treated municipalities ---
comparison = df[df["treated"] == 1][["CVEGEO", "municipio", "rank_composite"]].merge(
    df_pca[["CVEGEO", "rank_pca"]], on="CVEGEO"
).sort_values("rank_pca")

print("\n=== Rank comparison for treated municipalities ===")
print(comparison.to_string(index=False))


Correlation of PC1 with composite: 1.0000

RANK 1-61 (PCA)
     CVEGEO            municipio                          entidad  pc1_score  \
1899  25006            Culiac√°n                          Sinaloa  36.865608   
14    02004              Tijuana                  Baja California  26.026730   
704   15033  Ecatepec de Morelos                          M√©xico  18.701053   
244   08037              Ju√°rez                        Chihuahua  18.158491   
1701  21114               Puebla                           Puebla  14.560940   
...     ...                  ...                              ...        ...   
2300  30193             Veracruz  Veracruz de Ignacio de la Llave   2.822510   
991   19026            Guadalupe                      Nuevo Le√≥n   2.681781   
915   17006              Cuautla                          Morelos   2.671301   
372   11042    Valle de Santiago                       Guanajuato   2.621405   
615   14070             El Salto                          Ja

# LDA

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

# --- Identify 43 training treated from PCA ---
# These are treated municipalities that PCA ranked in the top N (captured)
# Adjust the rank cutoff to whatever threshold captured your 43
RANK_CUTOFF = 61  # replace with the rank where your 43 fall within

pca_captured_treated = df_pca[
    (df_pca["treated"] == 1) & (df_pca["rank_pca"] <= RANK_CUTOFF)
]["CVEGEO"]

strategic = df_pca[
    (df_pca["treated"] == 1) & (df_pca["rank_pca"] > RANK_CUTOFF)
]["CVEGEO"]

print(f"Training treated: {len(pca_captured_treated)}")
print(f"Strategic (excluded): {len(strategic)}")
print(f"Strategic CVEGEOs: {strategic.tolist()}")

# Verify counts
assert len(pca_captured_treated) == 43, f"Expected 43, got {len(pca_captured_treated)}"
assert len(strategic) == 18, f"Expected 18 strategic, got {len(strategic)}"

# --- Build LDA features ---
df_lda = df[["CVEGEO", "municipio", "entidad", "non_sampled", "treated"] + features].copy()
df_lda[features] = StandardScaler().fit_transform(df_lda[features])
df_lda["label"] = df_lda["CVEGEO"].isin(pca_captured_treated).astype(int)

X_train = df_lda[features]
y_train = df_lda["label"]

# --- Fit LDA ---
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

print("\n=== LDA coefficients ===")
for feat, coef in zip(features, lda.coef_[0]):
    print(f"  {feat}: {coef:.4f}")

# --- Score and rank all municipalities ---
df_lda["lda_score"] = lda.decision_function(df_lda[features])

if df_lda["lda_score"].corr(df["composite"]) < 0:
    df_lda["lda_score"] = -df_lda["lda_score"]

df_lda["rank_lda"] = df_lda["lda_score"].rank(ascending=False, method="min").astype(int)

# --- Coverage check ---
top61_lda = df_lda.nsmallest(61, "rank_lda")[
    ["CVEGEO", "municipio", "entidad", "lda_score", "rank_lda", "non_sampled", "treated"]
]
print("\nRANK 1-61 (LDA)")
print(top61_lda.to_string())
print(f"\nTreated in top 61 (LDA): {top61_lda['treated'].sum()} / 61")

# --- Three-way comparison for treated municipalities ---
comparison = df[df["treated"] == 1][["CVEGEO", "municipio", "rank_composite"]].merge(
    df_pca[["CVEGEO", "rank_pca"]], on="CVEGEO"
).merge(
    df_lda[["CVEGEO", "rank_lda"]], on="CVEGEO"
).sort_values("rank_lda")

print("\n=== Composite vs PCA vs LDA ranks (treated only) ===")
print(comparison.to_string(index=False))



Training treated: 43
Strategic (excluded): 18
Strategic CVEGEOs: ['02005', '02006', '04003', '06002', '10007', '12035', '15011', '15029', '15099', '19018', '19021', '20067', '21156', '23008', '24013', '31041', '31050', '32010']

=== LDA coefficients ===
  homicidio_doloso_prom_diario: 3.8705
  robo_veh_violencia_prom_diario: 2.7816
  n_inseguro_popw: 9.5863

RANK 1-61 (LDA)
     CVEGEO                    municipio                          entidad   lda_score  rank_lda  non_sampled  treated
1899  25006                    Culiac√°n                          Sinaloa  210.331596         1            0        1
14    02004                      Tijuana                  Baja California  193.091499         2            0        1
704   15033          Ecatepec de Morelos                          M√©xico  136.792912         3            0        1
244   08037                      Ju√°rez                        Chihuahua  127.201303         4            0        1
350   11020                      

In [25]:
comparison.to_csv("all_three_ranks.csv", index=False)